# [12-2강] CNN 학습 파이프라인 구성 - 실습

In [1]:
import torch
torch.set_num_threads(1)
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import random

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

torch.set_printoptions(precision=4, sci_mode=False)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


device: cpu


## 문제 1. toy image DataLoader 만들기

노트북 내부에서 만든 이미지/label Tensor를 TensorDataset과 DataLoader로 묶습니다.

In [2]:
def make_toy_images(n=48, size=8):
    # class 0: 세로선, class 1: 가로선
    x = torch.zeros(n, 1, size, size)
    y = torch.zeros(n, dtype=torch.long)
    for i in range(n):
        if i % 2 == 0:
            x[i, 0, :, 3:5] = 1.0
            y[i] = 0
        else:
            x[i, 0, 3:5, :] = 1.0
            y[i] = 1
    x += 0.05 * torch.randn_like(x)
    return x, y

images, labels = make_toy_images()
train_ds = TensorDataset(images[:40], labels[:40])
valid_ds = TensorDataset(images[40:], labels[40:])
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=8, shuffle=False)

# TODO: images와 labels를 TensorDataset으로 묶고 DataLoader를 만드세요.
dataset = TensorDataset(images, labels)
loader = DataLoader(dataset, batch_size=8, shuffle=True)
for batch in loader:
    xb, yb = batch
    print(xb.shape, yb.shape)
    break


torch.Size([8, 1, 8, 8]) torch.Size([8])


### 해설 및 실행 결과 해석

- DataLoader는 TensorDataset에서 sample을 꺼내 batch로 묶어 줍니다. image batch shape가 `[8, 1, 8, 8]`이면 CNN 입력으로 바로 사용할 수 있습니다.

## 문제 2. CNN 한 epoch 학습 함수 작성하기

mini-batch마다 forward, loss, backward, step 순서를 지키는 학습 함수를 완성합니다.

In [3]:
def make_toy_images(n=48, size=8):
    # class 0: 세로선, class 1: 가로선
    x = torch.zeros(n, 1, size, size)
    y = torch.zeros(n, dtype=torch.long)
    for i in range(n):
        if i % 2 == 0:
            x[i, 0, :, 3:5] = 1.0
            y[i] = 0
        else:
            x[i, 0, 3:5, :] = 1.0
            y[i] = 1
    x += 0.05 * torch.randn_like(x)
    return x, y

images, labels = make_toy_images()
train_ds = TensorDataset(images[:40], labels[:40])
valid_ds = TensorDataset(images[40:], labels[40:])
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=8, shuffle=False)

model = nn.Sequential(nn.Conv2d(1, 4, 3, padding=1), nn.ReLU(), nn.Flatten(), nn.Linear(4 * 8 * 8, 2)).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)

def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = loss_fn(logits, y)
        # TODO: gradient 초기화, 역전파, 파라미터 업데이트를 작성하세요.
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        total += x.size(0)
    return total_loss / total

print('train loss:', train_one_epoch(model, train_loader))


train loss: 0.31922377124428747


## 문제 3. validation accuracy 함수 작성하기

학습된 CNN을 평가 모드로 바꾸고 gradient 계산 없이 accuracy를 계산합니다.

In [6]:
def make_toy_images(n=48, size=8):
    # class 0: 세로선, class 1: 가로선
    x = torch.zeros(n, 1, size, size)
    y = torch.zeros(n, dtype=torch.long)
    for i in range(n):
        if i % 2 == 0:
            x[i, 0, :, 3:5] = 1.0
            y[i] = 0
        else:
            x[i, 0, 3:5, :] = 1.0
            y[i] = 1
    x += 0.05 * torch.randn_like(x)
    return x, y

images, labels = make_toy_images()
train_ds = TensorDataset(images[:40], labels[:40])
valid_ds = TensorDataset(images[40:], labels[40:])
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=8, shuffle=False)

model = nn.Sequential(nn.Conv2d(1, 4, 3, padding=1), nn.ReLU(), nn.Flatten(), nn.Linear(4 * 8 * 8, 2)).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)

def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += x.size(0)
    return total_loss / total, correct / total

def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = loss_fn(logits, y)
            total_loss += loss.item() * x.size(0)
            correct += (logits.argmax(dim=1) == y).sum().item()
            total += x.size(0)
    return total_loss / total, correct / total

for _ in range(3):
    train_one_epoch(model, train_loader, loss_fn, optimizer)

def accuracy(model, loader):
    model.eval()
    total = 0
    correct = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            correct += (pred.argmax(dim=1) == y).sum().item()
            total += x.size(0)
    return correct / total
print('valid acc:', accuracy(model, valid_loader))
    # TODO: 평가 모드와 no_grad를 사용해 accuracy를 계산하세요.
print('valid acc:', accuracy(model, valid_loader))


valid acc: 1.0
valid acc: 1.0


### 해설 및 실행 결과 해석

- validation에서는 파라미터를 업데이트하지 않습니다. accuracy가 높게 나오면 현재 toy data의 세로선/가로선 패턴을 모델이 어느 정도 구분한다는 뜻입니다.